In [40]:
import pandas as pd

def load_and_clean_data(file_path):
    df = pd.read_csv(file_path)

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df = df.sort_values(
        by=["patient_id", "timestamp"]
    )

    df = df.reset_index(drop=True)

    return df


In [41]:
def build_sequences(df):

    sequences = (
        df.groupby("patient_id")["event"]
        .apply(list)
        .tolist()
    )

    return sequences


In [42]:
from prefixspan import PrefixSpan

def mine_patterns(sequences, min_support=50):
    ps = PrefixSpan(sequences)
    patterns = ps.frequent(min_support)
    return patterns


In [43]:
df = load_and_clean_data(r"C:\Academics\vs code\Projects\Clinical-Trial-Mining\data\dataset.csv")

In [44]:
print(df["event"].value_counts())

event
Consent Signed                 6359
Visit Scheduled                6187
Sample Collected               5240
Data Entry Completed           4621
Lab Ordered                    4498
Lab Result Received            4489
Visit Completed                4475
Protocol Deviation             3379
Query Raised                   3319
Missed Visit                   1420
Visit Delayed                   925
Lab Result Delayed              752
Lab Missing                     749
Drug Dispensed                  684
Wrong Dose Recorded             679
Safety Review                   676
Vitals Recorded                 560
Medication Compliance Check     557
Query Resolved                  557
Data Entry Error                508
Monitoring Visit                499
Follow-up Visit                 493
Dose Delayed                    488
Consent Missing                 464
Audit Finding                   464
Name: count, dtype: int64


In [45]:
sequences = build_sequences(df)
sequences[:3][:]

[['Consent Signed',
  'Visit Scheduled',
  'Visit Completed',
  'Sample Collected',
  'Lab Ordered',
  'Data Entry Error',
  'Lab Result Received',
  'Data Entry Completed'],
 ['Consent Signed',
  'Visit Scheduled',
  'Visit Delayed',
  'Missed Visit',
  'Query Raised',
  'Protocol Deviation'],
 ['Consent Signed',
  'Visit Scheduled',
  'Visit Completed',
  'Sample Collected',
  'Lab Ordered',
  'Lab Result Received',
  'Data Entry Completed']]

In [46]:
patterns = mine_patterns(sequences, 100)
len(patterns)

980

In [47]:
print(patterns[:20])

[(6359, ['Consent Signed']), (6187, ['Consent Signed', 'Visit Scheduled']), (4353, ['Consent Signed', 'Visit Scheduled', 'Visit Completed']), (4220, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected']), (4097, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered']), (3965, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received']), (3965, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received', 'Data Entry Completed']), (4097, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Data Entry Completed']), (104, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Query Resolved']), (104, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Query Resolved', 'Data Entry Completed']), (126, ['Consen

In [48]:
for support, pattern in patterns[:20]:
    print(f"Support: {support}, Length: {len(pattern)}, Pattern: {pattern}")

Support: 6359, Length: 1, Pattern: ['Consent Signed']
Support: 6187, Length: 2, Pattern: ['Consent Signed', 'Visit Scheduled']
Support: 4353, Length: 3, Pattern: ['Consent Signed', 'Visit Scheduled', 'Visit Completed']
Support: 4220, Length: 4, Pattern: ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected']
Support: 4097, Length: 5, Pattern: ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered']
Support: 3965, Length: 6, Pattern: ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received']
Support: 3965, Length: 7, Pattern: ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received', 'Data Entry Completed']
Support: 4097, Length: 6, Pattern: ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Data Entry Completed']
Support: 104, Length: 6, Pattern: ['Consent Signed', 'Visit Schedul

In [49]:
lengths = [len(pattern) for support, pattern in patterns]

print("Min Length:", min(lengths))
print("Max Length:", max(lengths))
print("Average Length:", sum(lengths)/len(lengths))

Min Length: 1
Max Length: 7
Average Length: 3.7510204081632654


In [50]:
filtered_patterns = [(support, pattern) for support, pattern in patterns if len(pattern) >= 2]

In [51]:
len(filtered_patterns)

955

In [52]:
lengths = [len(patterns) for support, pattern in patterns]

print("Min Length:", min(lengths))
print("Max Length:", max(lengths))
print("Average Length:", sum(lengths)/len(lengths))

Min Length: 980
Max Length: 980
Average Length: 980.0


In [53]:
print(filtered_patterns[:5])

[(6187, ['Consent Signed', 'Visit Scheduled']), (4353, ['Consent Signed', 'Visit Scheduled', 'Visit Completed']), (4220, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected']), (4097, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered']), (3965, ['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received'])]


In [54]:
for support, pattern in filtered_patterns[:20]:
    print(
        f"Support={support}, "
        f"Length={len(pattern)}, "
        f"Pattern={pattern}"
    )

Support=6187, Length=2, Pattern=['Consent Signed', 'Visit Scheduled']
Support=4353, Length=3, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed']
Support=4220, Length=4, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected']
Support=4097, Length=5, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered']
Support=3965, Length=6, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received']
Support=3965, Length=7, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Lab Result Received', 'Data Entry Completed']
Support=4097, Length=6, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Data Entry Completed']
Support=104, Length=6, Pattern=['Consent Signed', 'Visit Scheduled', 'Visit Completed', 'Sample Collected', 'Lab Ordered', 'Query Resolved']
S

In [59]:
deviation_patterns = [
    (support, pattern)
    for support, pattern in filtered_patterns
    if pattern[-1] == "Protocol Deviation"
]

In [60]:
deviation_patterns.sort(
    key=lambda x: x[0],
    reverse=True
)

In [61]:
for support, pattern in deviation_patterns[:20]:
    print(
        f"Support={support}, "
        f"Length={len(pattern)}, "
        f"Pattern={pattern}"
    )

Support=3319, Length=2, Pattern=['Query Raised', 'Protocol Deviation']
Support=1738, Length=2, Pattern=['Consent Signed', 'Protocol Deviation']
Support=1690, Length=3, Pattern=['Consent Signed', 'Query Raised', 'Protocol Deviation']
Support=1688, Length=3, Pattern=['Consent Signed', 'Visit Scheduled', 'Protocol Deviation']
Support=1688, Length=2, Pattern=['Visit Scheduled', 'Protocol Deviation']
Support=1640, Length=4, Pattern=['Consent Signed', 'Visit Scheduled', 'Query Raised', 'Protocol Deviation']
Support=1640, Length=3, Pattern=['Visit Scheduled', 'Query Raised', 'Protocol Deviation']
Support=1420, Length=2, Pattern=['Missed Visit', 'Protocol Deviation']
Support=1390, Length=3, Pattern=['Missed Visit', 'Query Raised', 'Protocol Deviation']
Support=936, Length=3, Pattern=['Consent Signed', 'Missed Visit', 'Protocol Deviation']
Support=925, Length=3, Pattern=['Consent Signed', 'Visit Delayed', 'Protocol Deviation']
Support=925, Length=2, Pattern=['Visit Delayed', 'Protocol Deviation

In [62]:
total_sequences = len(sequences)

patterns_with_pct = [
    (
        support,
        round(support * 100 / total_sequences, 2),
        pattern
    )
    for support, pattern in deviation_patterns
]

In [68]:
for support, pct, pattern in patterns_with_pct[:10]:
    print(
        f"{pct}% | {support} | {pattern}"
    )

41.49% | 3319 | ['Query Raised', 'Protocol Deviation']
21.73% | 1738 | ['Consent Signed', 'Protocol Deviation']
21.12% | 1690 | ['Consent Signed', 'Query Raised', 'Protocol Deviation']
21.1% | 1688 | ['Consent Signed', 'Visit Scheduled', 'Protocol Deviation']
21.1% | 1688 | ['Visit Scheduled', 'Protocol Deviation']
20.5% | 1640 | ['Consent Signed', 'Visit Scheduled', 'Query Raised', 'Protocol Deviation']
20.5% | 1640 | ['Visit Scheduled', 'Query Raised', 'Protocol Deviation']
17.75% | 1420 | ['Missed Visit', 'Protocol Deviation']
17.38% | 1390 | ['Missed Visit', 'Query Raised', 'Protocol Deviation']
11.7% | 936 | ['Consent Signed', 'Missed Visit', 'Protocol Deviation']
